# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane 2 continuation** — same refresh-priority ranking decision framed in `w01`/`w02`, now built on the real warehouse panel instead of the static starter CSV.

- **One row =** one `(client_hash_id, content_hash_id, report_date)` triple in `fact_content_daily_performance` — one content item's GSC performance on one calendar day for one client. Verified in Query 1 below.
- **Table(s) used:** `fact_content_daily_performance`, restricted to the **`month=2026-03`** partition only (a mid-panel month) — never the full 79M-row scan, and never `fact_content_daily_performance_sample`. That table *is* the sealed final month (June 2026); using it here would leak the held-out test month into what's supposed to be ordinary training-month practice.
- **Time window:** all of March 2026, split in half for this exercise — **days 1–15** = the feature window (known at the decision moment), **days 16–31** = the outcome window (what happened next). A compressed stand-in for the pipeline's real last30/prev30 split, since I'm deliberately querying only one month's partition.
- **Decision moment:** March 16, 2026 — a reviewer picking pages to check on this date, using only what's observable through March 15.
- **What I'd predict/rank (label or proxy):** the same Lane 2 target as `w01`/`w02` — is this page's traffic **declining** — redefined for this slice as `is_declining = impressions(days 16-31) < 0.8 × impressions(days 1-15)`. This is a **proxy**, not an observed future outcome: both halves sit inside the same queried month, so nothing here required waiting to see what actually happens next — the same caveat I flagged in `w02` for the starter CSV's `trend_direction`.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy (never a feature):** `is_declining`, defined above — built from `gsc_impressions` summed separately over days 1–15 vs 16–31. Anything computed from the days-16–31 window is off-limits as a feature for the same reason.

**Context (grouping only, never a feature):** `client_hash_id`, `content_hash_id`, `report_date` — used to group, join, and split (client-holdout), never fed to a model.

**One thing I deliberately exclude:** every GA4-derived metric (pageviews, sessions, engagement). `fact_content_daily_performance` zero-fills GA4 columns before a client's `ga4_data_start`, flagged by `ga4_data_available`. Mixing those zeros in without checking the flag would silently teach a model "client is new" instead of "page has no engagement." Query 3 below measures how much of this one month survives an `IS TRUE` filter on that flag — until I've checked it per-client (not just in aggregate), this slice stays GSC-only.

**Features** (the fourth bucket) are enumerated where they're built, in section 3 below — each with a one-line "knowable at the decision moment because…" justification.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib python-dotenv

import os, getpass
import duckdb
import pandas as pd
from dotenv import load_dotenv

# Local dev: pulls HF_TOKEN from .env if present (gitignored -- never commit it).
# Falls back to env var, then to a safe interactive prompt if neither is set.
load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Mid-panel month only -- never the full 79M-row scan, and never the _sample table
# (that table IS the sealed final month, June 2026 -- using it here would leak the
# held-out test month into what's supposed to be ordinary training-month practice).
MONTH = "2026-03"
fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

print(f"Querying month={MONTH} only -- partition pruning means this never touches the other 16 months.")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\iamlu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Querying month=2026-03 only -- partition pruning means this never touches the other 16 months.


In [2]:
# Query 1 -- grain: one row really is one (client, content, day)
dupes = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {fact_month}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"duplicate (client, content, day) combinations found: {len(dupes)}  -- 0 means the grain holds")
dupes

duplicate (client, content, day) combinations found: 0  -- 0 means the grain holds


,client_hash_id,content_hash_id,report_date,c


In [3]:
# Query 2 -- row count and date span for this slice
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {fact_month}
""").df()
span

,n_rows,n_clients,n_content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [4]:
# Query 3 -- availability: how much of this month survives an IS TRUE filter
# on ga4_data_available? (Rows before a client's ga4_data_start are zero-filled,
# not genuinely "no engagement" -- this is why the contract above excludes GA4
# metrics from this slice until checked per-client.)
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {fact_month}
""").df()
availability["ga4_available_pct"] = 100 * availability["ga4_available_rows"] / availability["total_rows"]
print("Rows that survive an `IS TRUE` filter on ga4_data_available:")
availability

Rows that survive an `IS TRUE` filter on ga4_data_available:


,total_rows,ga4_available_rows,ga4_available_pct
0,9841378,413966.0,4.206382


### 5 features (max), built from days 1–15 of March 2026 only

Decision moment: **March 16, 2026**. Every feature below uses only `report_date <= '2026-03-15'` — fully in the past relative to the decision moment, so each is "knowable" at the point a reviewer would actually be looking at the queue.

1. `imp_prev15` — total GSC impressions, days 1–15. Knowable at the decision moment because it's a straight sum over calendar days already elapsed by March 16.
2. `clicks_prev15` — total GSC clicks, days 1–15. Same reasoning: fully in the past.
3. `ctr_prev15` — `clicks_prev15 / imp_prev15`. Derived entirely from the two features above, so it inherits the same availability.
4. `avg_position_prev15` — mean GSC position, days 1–15. Same reasoning.
5. `active_days_prev15` — count of days with impressions > 0, days 1–15 (max 15). Knowable because it's a count of already-elapsed days, not a forecast.

`imp_last15` (total impressions, days 16–31) is pulled too, but only to build the label — it is **not** one of the 5 features, and gets used as the deliberate leak below.

In [5]:
feat = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{MONTH}-15' THEN gsc_impressions ELSE 0 END) AS imp_prev15,
        SUM(CASE WHEN report_date <= DATE '{MONTH}-15' THEN gsc_clicks ELSE 0 END)      AS clicks_prev15,
        AVG(CASE WHEN report_date <= DATE '{MONTH}-15' THEN gsc_avg_position END)       AS avg_position_prev15,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '{MONTH}-15' AND gsc_impressions > 0
                             THEN report_date END)                                       AS active_days_prev15,
        SUM(CASE WHEN report_date > DATE '{MONTH}-15' THEN gsc_impressions ELSE 0 END)  AS imp_last15
    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_prev15 >= 10
""").df()

feat["ctr_prev15"] = feat["clicks_prev15"] / feat["imp_prev15"].replace(0, pd.NA)
feat["is_declining"] = (feat["imp_last15"] < 0.8 * feat["imp_prev15"]).astype(int)

print(f"{len(feat):,} content items with enough prev-window history (imp_prev15 >= 10)")
feat.head()

120,513 content items with enough prev-window history (imp_prev15 >= 10)


,client_hash_id,content_hash_id,imp_prev15,clicks_prev15,avg_position_prev15,active_days_prev15,imp_last15,ctr_prev15,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,15,20.0,0.000000,1
1,client_62f4a7e64f5e0096,content_3467af9d4681980d,16.0,0.0,3.883333,10,6.0,0.000000,1
2,client_62f4a7e64f5e0096,content_cf6b1cadbfd9eab1,233.0,0.0,5.036708,15,38.0,0.000000,1
3,client_62f4a7e64f5e0096,content_fb1a30c7e8493b76,977.0,2.0,16.201956,15,1955.0,0.002047,0
4,client_62f4a7e64f5e0096,content_0bf2def807f81ee1,242.0,0.0,34.087287,15,141.0,0.000000,1


### The trap: add a label-derived column on purpose

`imp_last15` is literally the outcome-window number used to build `is_declining` — it has no business being a feature. Adding it on purpose below, watching the score jump, then removing it — the leakage lesson from notebook 02, on real warehouse data.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_data = feat.dropna(subset=[
    "imp_prev15", "clicks_prev15", "ctr_prev15", "avg_position_prev15",
    "active_days_prev15", "is_declining",
])

honest_features = ["imp_prev15", "clicks_prev15", "ctr_prev15", "avg_position_prev15", "active_days_prev15"]
leaky_features = honest_features + ["imp_last15"]  # <- the label-derived column, added on purpose

def quick_auc(cols):
    X, y = model_data[cols], model_data["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_features)
leaky_auc = quick_auc(leaky_features)

print(f"honest AUC (5 features, no leak):        {honest_auc:.3f}")
print(f"leaky AUC (+ imp_last15, the label leak): {leaky_auc:.3f}  <- jumps toward 1.0, same lesson as notebook 02")
print()
print(f"KEEPING the honest number: {honest_auc:.3f}. imp_last15 is not part of the kept feature set.")

honest AUC (5 features, no leak):        0.585
leaky AUC (+ imp_last15, the label leak): 1.000  <- jumps toward 1.0, same lesson as notebook 02

KEEPING the honest number: 0.585. imp_last15 is not part of the kept feature set.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of this slice:** a single mid-panel month (March 2026) split in half (15/15 days) is a compressed stand-in for the pipeline's real last30/prev30 comparison. Fifteen days of GSC history is noisy for lower-traffic pages — small denominators swing `ctr_prev15` and `avg_position_prev15` a lot — and this slice says nothing about clients whose `gsc_data_start` falls after March 2026 (they're simply absent here, not zero) or about months outside March (holiday traffic, algorithm updates, seasonal swings). Any score or threshold measured here is specific to this one month and this one half/half split — it has to be re-earned on the full multi-month panel with a proper 30/30 or 60/60 window before it means anything for the capstone.

## Self-check

Before you submit, confirm each line honestly:

- [ yep] Every section above is filled — markdown thinking AND the code that backs it
- [yep ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yep] No client names, URLs, or private queries anywhere
- [yep ] My claims use careful words: observed, measured, directional, decision-support
- [yep ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.